In [9]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score

pd.set_option('display.max_columns', None)

In [11]:
df_clean = pd.read_pickle('data/clean_transactions.pkl')
print(df_clean.shape)

# The last date in the data. Everything is measured relative to it.
max_date = df_clean['InvoiceDate'].max()
print(max_date)                        

HORIZON = 90        # "will the customer buy again within 90 days?"
QUIET_DAYS = 60     # a customer is "quiet" after 60 days without a purchase (see Part C)

(805549, 9)
2011-12-09 12:50:00


**Helper Function**

In [12]:
def build_dataset(df, cutoff, horizon_days):
    past = df[df['InvoiceDate'] <= cutoff]
    future = df[(df['InvoiceDate'] > cutoff) &
                (df['InvoiceDate'] <= cutoff + pd.Timedelta(days=horizon_days))]
    feat = past.groupby('Customer ID').agg(
        Recency=('InvoiceDate', lambda x: (cutoff - x.max()).days),
        Frequency=('Invoice', 'nunique'),
        Monetary=('TotalPrice', 'sum'),
        FirstPurchase=('InvoiceDate', 'min')).reset_index()
    feat['label'] = feat['Customer ID'].isin(set(future['Customer ID'])).astype(int)
    return feat

def prep(feat, cap=None):
    spend = feat['Monetary'].clip(upper=cap) if cap else feat['Monetary']
    return pd.DataFrame({'Recency': feat['Recency'],
                         'log_Frequency': np.log1p(feat['Frequency']),
                         'log_Monetary': np.log1p(spend)})